<a href="https://colab.research.google.com/github/KJ0211/Machine-learning-portfolio/blob/main/Knowledge%20Tracing%20on%20Duolingo%20Learner%20Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())  # not essential this time, but fine to leave the runtime on GPU

GPU available: True


In [3]:
from google.colab import files
uploaded = files.upload()  # pick en_es.slam.20190204.train from your computer

Saving ._CHANGELOG.md to ._CHANGELOG.md
Saving ._CITING.md to ._CITING.md
Saving ._en_es.slam.20190204.dev to ._en_es.slam.20190204.dev
Saving ._en_es.slam.20190204.dev.key to ._en_es.slam.20190204.dev.key
Saving ._en_es.slam.20190204.test to ._en_es.slam.20190204.test
Saving ._en_es.slam.20190204.test.key to ._en_es.slam.20190204.test.key
Saving ._en_es.slam.20190204.train to ._en_es.slam.20190204.train
Saving ._LICENSE.md to ._LICENSE.md
Saving CHANGELOG.md to CHANGELOG.md
Saving CITING.md to CITING.md
Saving en_es.slam.20190204.dev to en_es.slam.20190204.dev
Saving en_es.slam.20190204.dev.key to en_es.slam.20190204.dev.key
Saving en_es.slam.20190204.test to en_es.slam.20190204.test
Saving en_es.slam.20190204.test.key to en_es.slam.20190204.test.key
Saving en_es.slam.20190204.train to en_es.slam.20190204.train
Saving LICENSE.md to LICENSE.md


In [4]:
with open("en_es.slam.20190204.train") as f:
    train_raw_lines = [next(f) for _ in range(15)]

for line in train_raw_lines:
    print(repr(line))


'# prompt:Yo soy un niño.\n'
'# user:XEinXf5+  countries:CO  days:0.003  client:web  session:lesson  format:reverse_translate  time:9\n'
'DRihrVmh0101  I             PRON    Case=Nom|Number=Sing|Person=1|PronType=Prs|fPOS=PRON++PRP               nsubj        4  0\n'
'DRihrVmh0102  am            VERB    Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbForm=Fin|fPOS=VERB++VBP    cop          4  0\n'
'DRihrVmh0103  a             DET     Definite=Ind|PronType=Art|fPOS=DET++DT                                  det          4  0\n'
'DRihrVmh0104  boy           NOUN    Number=Sing|fPOS=NOUN++NN                                               ROOT         0  0\n'
'\n'
'# prompt:Yo soy de México.\n'
'# user:XEinXf5+  countries:CO  days:0.005  client:web  session:lesson  format:reverse_translate  time:12\n'
'TOeLHxLS0101  I             PRON    Case=Nom|Number=Sing|Person=1|PronType=Prs|fPOS=PRON++PRP               nsubj        4  0\n'
'TOeLHxLS0102  am            VERB    Mood=Ind|Number=Sing|Person=1|T

In [5]:
def parse_slam_file(path, has_labels):
    exercises = []
    current_meta = None
    current_tokens = []

    with open(path) as f:
        for raw_line in f:
            line = raw_line.rstrip("\n")
            if line.startswith("# prompt:"):
                continue  # source-language sentence; not used in this first pass
            elif line.startswith("# user:"):
                if current_meta is not None:
                    exercises.append((current_meta, current_tokens))
                fields = line[2:].strip().split()
                current_meta = dict(kv.split(":", 1) for kv in fields if ":" in kv)
                current_tokens = []
            elif line.strip() == "":
                continue
            else:
                current_tokens.append(line.split())

        if current_meta is not None:
            exercises.append((current_meta, current_tokens))
    return exercises


def exercises_to_df(exercises, has_labels):
    rows = []
    for meta, tokens in exercises:
        for tok in tokens:
            row = dict(meta)
            row["instance_id"] = tok[0]
            row["token"] = tok[1]
            row["pos"] = tok[2]
            row["morph"] = tok[3]
            row["dep_label"] = tok[4]
            row["dep_head"] = tok[5]
            if has_labels:
                row["label"] = int(tok[6])
            rows.append(row)
    return pd.DataFrame(rows)

In [7]:
import pandas as pd
dev_exercises = parse_slam_file("en_es.slam.20190204.dev", has_labels=False)
dev_df = exercises_to_df(dev_exercises, has_labels=False)

dev_key = pd.read_csv("en_es.slam.20190204.dev.key", sep=" ", header=None, names=["instance_id", "label"])
dev_df = dev_df.merge(dev_key, on="instance_id", how="left")

print(dev_df.shape)
print(dev_df["label"].isna().sum())  # must be 0 -- every token should get a label from the key file
print(dev_df["label"].value_counts(normalize=True))

(387374, 14)
0
label
0    0.857094
1    0.142906
Name: proportion, dtype: float64


In [8]:
train_exercises = parse_slam_file("en_es.slam.20190204.train", has_labels=True)
train_df = exercises_to_df(train_exercises, has_labels=True)
print(train_df.shape)
print(train_df["label"].value_counts(normalize=True))

(2622957, 14)
label
0    0.873887
1    0.126113
Name: proportion, dtype: float64


In [9]:
train_df["days"] = train_df["days"].astype(float)
train_df = train_df.sort_values(["user", "days"])

train_df["past_attempts"] = train_df.groupby("user").cumcount()
train_df["past_correct_cumsum"] = (
    train_df.groupby("user")["label"]
    .apply(lambda s: (1 - s).cumsum().shift(1))  # (1 - label) because label=1 means ERROR, so "correct" = 1 - label
    .reset_index(level=0, drop=True)
)
train_df["historical_accuracy"] = (
    train_df["past_correct_cumsum"] / train_df["past_attempts"]
).fillna(0.5)  # 0.5 for a user's very first attempt, where there's no history yet

In [10]:
sample_user = train_df["user"].iloc[0]
print(train_df[train_df["user"] == sample_user][["days", "label", "past_attempts", "historical_accuracy"]].head(10))

          days  label  past_attempts  historical_accuracy
2118170  0.004      0              0             0.500000
2118171  0.004      1              1             1.000000
2118172  0.004      0              2             0.500000
2118173  0.004      0              3             0.666667
2118174  1.059      0              4             0.750000
2118175  1.059      0              5             0.800000
2118176  1.059      0              6             0.833333
2118177  1.059      0              7             0.857143
2118178  1.059      0              8             0.875000
2118179  1.059      0              9             0.888889


In [11]:
from sklearn.metrics import roc_auc_score, f1_score

majority_class = train_df["label"].mode()[0]
baseline_pred = [majority_class] * len(train_df)
print("Majority-class baseline F1:", f1_score(train_df["label"], baseline_pred))

hist_baseline_pred = (train_df["historical_accuracy"] < 0.5).astype(int)  # predicting ERROR (label=1) when history is poor
print("Historical-accuracy-only baseline F1:", f1_score(train_df["label"], hist_baseline_pred))
print("Historical-accuracy-only baseline AUC:", roc_auc_score(train_df["label"], 1 - train_df["historical_accuracy"]))

Majority-class baseline F1: 0.0
Historical-accuracy-only baseline F1: 0.015621461181771585
Historical-accuracy-only baseline AUC: 0.6566273781882976


In [12]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

feature_cols = ["historical_accuracy", "past_attempts", "pos", "format", "days"]

model_df = train_df.copy()
for col in ["pos", "format"]:
    model_df[col] = model_df[col].astype("category")

train_split, test_split = train_test_split(
    model_df, test_size=0.2, random_state=42, stratify=model_df["label"]
)

clf = HistGradientBoostingClassifier(categorical_features=["pos", "format"], random_state=42)
clf.fit(train_split[feature_cols], train_split["label"])

pred_proba = clf.predict_proba(test_split[feature_cols])[:, 1]
pred = (pred_proba > 0.5).astype(int)

print("Model F1:", f1_score(test_split["label"], pred))
print("Model AUC:", roc_auc_score(test_split["label"], pred_proba))

Model F1: 0.10966172894096839
Model AUC: 0.7322150562276606


In [13]:
from sklearn.metrics import precision_recall_curve, classification_report

precision, recall, thresholds = precision_recall_curve(test_split["label"], pred_proba)
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)
print("Best F1 at that threshold:", f1_scores[best_idx])

pred_tuned = (pred_proba > best_threshold).astype(int)
print(classification_report(test_split["label"], pred_tuned))

Best threshold: 0.16403394155553847
Best F1 at that threshold: 0.3471680530017553
              precision    recall  f1-score   support

           0       0.92      0.78      0.84    458434
           1       0.26      0.53      0.35     66158

    accuracy                           0.75    524592
   macro avg       0.59      0.65      0.60    524592
weighted avg       0.84      0.75      0.78    524592



In [14]:
from sklearn.metrics import roc_auc_score, f1_score

majority_class = train_df["label"].mode()[0]
baseline_pred = [majority_class] * len(train_df)
print("Majority-class baseline F1:", f1_score(train_df["label"], baseline_pred))

print("Historical-accuracy-only baseline AUC:", roc_auc_score(train_df["label"], 1 - train_df["historical_accuracy"]))

Majority-class baseline F1: 0.0
Historical-accuracy-only baseline AUC: 0.6566273781882976
